In [0]:
%run ../00_setup

In [0]:
def filter_by_watermark(df: DataFrame, watermark_column: str, last_watermark) -> DataFrame:
    """
    Se queda solo con los registros cuyo campo de fecha (ej. meta.updatedAt)
    es POSTERIOR al último watermark guardado en control.bronze_load_config.

    Si last_watermark es None (primera ejecución de esta entidad),
    no se filtra nada — se acepta todo el snapshot como punto de partida.
    """
    df_con_watermark = df.withColumn(
        "_watermark_value", F.to_timestamp(F.col(watermark_column))
    )

    if last_watermark is None:
        return df_con_watermark

    return df_con_watermark.filter(F.col("_watermark_value") > F.lit(last_watermark))


def compute_new_watermark(df_filtrado: DataFrame):
    """
    Calcula el nuevo watermark a partir de los registros que SÍ pasaron el
    filtro en esta ejecución. Si no hay registros nuevos, devuelve None
    (y el motor debe conservar el watermark anterior sin modificarlo).
    """
    resultado = df_filtrado.agg(F.max("_watermark_value").alias("max_wm")).collect()[0]
    return resultado.max_wm  # puede ser None si df_filtrado está vacío